# 05 — Vibrational Analysis & Thermochemistry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/05_vibrational_analysis.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- Understand the harmonic approximation and mass-weighted Hessian
- Compute vibrational frequencies and normal modes with PySCF
- Identify IR-active vibrations and simulate IR spectra
- Calculate zero-point energy (ZPE) corrections
- Extract thermochemical properties (ΔH, ΔG, S) from frequency calculations
- Recognize imaginary frequencies and their meaning
- Apply vibrational scaling factors to improve accuracy

## 1. Theory: Normal Mode Analysis

### 1.1 The Hessian Matrix

The Hessian (second derivative matrix) at a stationary point defines the curvature of the PES:

$$H_{ij} = \frac{\partial^2 E}{\partial q_i \partial q_j}$$

For a molecule with $N$ atoms, the Cartesian Hessian is a $3N \times 3N$ matrix.

### 1.2 Normal Modes

The **mass-weighted Hessian** $\tilde{H}$ is formed by:
$$\tilde{H}_{ij} = \frac{H_{ij}}{\sqrt{m_i m_j}}$$

Diagonalization yields the **normal mode frequencies** $\omega_k$ and eigenvectors $\mathbf{Q}_k$:
$$\tilde{H} \mathbf{Q}_k = \omega_k^2 \mathbf{Q}_k$$

For a non-linear molecule with $N$ atoms:
- 3 translational modes: $\omega = 0$
- 3 rotational modes: $\omega = 0$ (2 for linear)
- $3N - 6$ vibrational modes (3N-5 for linear)

Frequencies in wavenumbers:
$$\tilde{\nu}_k = \frac{\omega_k}{2\pi c}$$

### 1.3 The Harmonic Approximation

In the harmonic approximation, each normal mode behaves as a quantum harmonic oscillator:
$$E_n = \hbar\omega_k \left(n + \frac{1}{2}\right)$$

The **zero-point energy (ZPE)** is the ground-state vibrational energy:
$$E_{ZPE} = \frac{1}{2}\sum_k \hbar\omega_k$$

### 1.4 Thermochemistry

From frequencies and the optimized geometry, we compute partition functions:

$$q_{total} = q_{trans} \cdot q_{rot} \cdot q_{vib} \cdot q_{elec}$$

Thermodynamic quantities (at temperature $T$, pressure $P$):
- **Enthalpy**: $H = E_{elec} + E_{ZPE} + \Delta E_{thermal} + k_B T$
- **Entropy**: $S = S_{trans} + S_{rot} + S_{vib} + S_{elec}$
- **Gibbs free energy**: $G = H - TS$

### 1.5 IR Spectroscopy

An IR vibration is **active** if it changes the **dipole moment**:
$$I_{IR} \propto \left|\frac{\partial \boldsymbol{\mu}}{\partial Q_k}\right|^2$$

For water (C\u2082v symmetry):
- Symmetric stretch (A\u2081): **IR active** (changes \u03bc in z-direction)
- Asymmetric stretch (B\u2082): **IR active**
- Bending (A\u2081): **IR active**

Compare with CO\u2082 (D\u221eh): symmetric stretch is IR **inactive** (no change in \u03bc).

In [ ]:
%%time
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 05: Vibrational Analysis
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyscf import gto, dft
from pyscf.hessian import rks as rks_hess
from pyscf.hessian import thermo

# ------------------------------------------------------------------
# Hessian and Vibrational Frequencies for H₂O
# ------------------------------------------------------------------
# B3LYP/def2-SVP optimized geometry of water

mol = gto.Mole()
mol.atom = '''
O   0.000000   0.000000   0.117176
H   0.000000   0.757001  -0.468704
H   0.000000  -0.757001  -0.468704
'''
mol.basis = 'def2-SVP'
mol.verbose = 0
mol.build()

# Step 1: Run the DFT calculation to get the ground-state density
mf = dft.RKS(mol)
mf.xc = 'B3LYP'
mf.verbose = 0
mf.kernel()

# Step 2: Compute the analytical Hessian matrix
hess = rks_hess.Hessian(mf)
hess.verbose = 0
h = hess.kernel()

# Step 3: Perform normal mode analysis
freq_info = thermo.harmonic_analysis(mol, h)

freqs_cm = freq_info['freq_wavenumber']   # wavenumbers (cm^-1)
freqs_au = freq_info['freq_au']           # atomic units (Hartree)
modes    = freq_info['norm_mode']         # normal mode displacements

print('=' * 60)
print('  Vibrational Frequencies for H₂O at B3LYP/def2-SVP')
print('=' * 60)
print(f"  {'Mode':6s} {'Freq (cm⁻¹)':>14s}  {'Type':20s}")
print(f"  {'-'*6} {'-'*14}  {'-'*20}")

mode_labels = {
    0: 'Translation (x)',
    1: 'Translation (y)',
    2: 'Translation (z)',
    3: 'Rotation (x)',
    4: 'Rotation (y)',
    5: 'Rotation (z)',
    6: 'Bending',
    7: 'Symmetric stretch',
    8: 'Asymmetric stretch',
}

for i, freq in enumerate(freqs_cm):
    label = mode_labels.get(i, f'Mode {i+1}')
    tag = '' if abs(freq) < 10 else ('IMAGINARY ⚠️' if freq < 0 else 'IR active ✓')
    print(f"  {i+1:6d} {freq:14.2f}  {label:20s}  {tag}")

print(f'\nNon-zero vibrational frequencies (cm⁻¹):')
vib_freqs = [f for f in freqs_cm if abs(f) > 50]
for f in vib_freqs:
    print(f'  {f:.1f} cm⁻¹')

print(f'\nExperimental H₂O frequencies (cm⁻¹):')
print(f'  1595 cm⁻¹ (bending)')
print(f'  3657 cm⁻¹ (symmetric stretch)')
print(f'  3756 cm⁻¹ (asymmetric stretch)')

# Compute ZPE
zpe_au = 0.5 * np.sum([f for f in freqs_au if f > 0])
zpe_ev = zpe_au * 27.2114
zpe_kcal = zpe_au * 627.509
print(f'\nZero-Point Energy (ZPE):')
print(f'  ZPE = {zpe_au:.6f} Ha = {zpe_ev:.4f} eV = {zpe_kcal:.2f} kcal/mol')

In [ ]:
# ------------------------------------------------------------------
# Simulated IR Spectrum with Gaussian Broadening
# ------------------------------------------------------------------
# We use the calculated frequencies with approximate intensities.
# In a production calculation, intensities come from the dipole
# derivative (not computed here, so we use model intensities).

import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import numpy as np

# H2O at B3LYP/def2-SVP — frequencies and approximate intensities
# Intensities (km/mol) are approximate; production values need dipole derivatives
freq_data = [
    (1618,  65.9, 'Bending (scissors)'),
    (3815,  10.2, 'Symmetric O-H stretch'),
    (3905,  52.1, 'Asymmetric O-H stretch'),
]

# Plot range
nu = np.linspace(400, 4200, 3000)   # wavenumber axis

sigma = 25.0   # Gaussian width (cm^-1) for instrument broadening

# Build spectrum: sum of Gaussians
spectrum = np.zeros_like(nu)
for freq, intensity, label in freq_data:
    spectrum += intensity * np.exp(-0.5 * ((nu - freq) / sigma)**2)

fig, ax = plt.subplots(figsize=(10, 5))

# Broadened envelope
ax.plot(nu, spectrum, '-', color='steelblue', linewidth=2.5, label='Simulated IR spectrum')
ax.fill_between(nu, spectrum, alpha=0.15, color='steelblue')

# Stick spectrum
for freq, intensity, label in freq_data:
    ax.vlines(freq, 0, intensity, color='coral', linewidth=3, alpha=0.8)
    ax.text(freq, intensity + 2, f'{freq:.0f}', ha='center', va='bottom',
            fontsize=9, color='coral', fontweight='bold')
    ax.text(freq, -8, label, ha='center', va='top', fontsize=8.5,
            color='gray', rotation=30)

# Add experimental peaks (dashed)
exp_freqs = [1595, 3657, 3756]
exp_labels = ['1595', '3657', '3756']
for ef, el in zip(exp_freqs, exp_labels):
    ax.axvline(x=ef, color='seagreen', linestyle='--', alpha=0.6, linewidth=1.5)
    ax.text(ef, max(spectrum)*0.85, f'Exp:\n{el}', ha='center', va='center',
            fontsize=8, color='seagreen')

ax.set_xlim(400, 4200)
ax.set_ylim(-15, max(spectrum) * 1.25)
ax.set_xlabel('Wavenumber (cm⁻¹)', fontsize=12)
ax.set_ylabel('Intensity (km/mol)', fontsize=12)
ax.set_title('Simulated IR Spectrum of H₂O\nB3LYP/def2-SVP (Gaussian broadening σ=25 cm⁻¹)', fontsize=12)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='steelblue', linewidth=2.5, label='Simulated (broadened)'),
    Line2D([0], [0], color='coral', linewidth=3, label='Stick spectrum'),
    Line2D([0], [0], color='seagreen', linestyle='--', label='Experimental'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Note: A scaling factor of ~0.97 would shift computed frequencies')
print('closer to experimental values (systematic overestimation in harmonic approx)')

In [ ]:
# ------------------------------------------------------------------
# Thermochemistry: ZPE, Enthalpy, Gibbs Free Energy
# ------------------------------------------------------------------
# Using PySCF's thermo module for full thermochemical analysis

from pyscf.hessian import thermo as thermo_mod
import numpy as np

# Use freq_info and mf from the frequency cell above
T = 298.15    # Temperature in K
P = 101325.0  # Pressure in Pa (1 atm)

# Run thermochemical analysis
thermo_result = thermo_mod.thermo(mf, freq_info['freq_au'], T, P)

print('=' * 60)
print(f'  Thermochemical Analysis at T = {T} K, P = 1 atm')
print('=' * 60)
print(f'  Electronic energy (SCF):   {mf.e_tot:15.8f} Ha')
print(f'                             {mf.e_tot * 27.2114:15.4f} eV')
print()

# Extract thermochemical quantities
# ZPE
zpe_ha = thermo_result['ZPE'][0]
print(f'  Zero-Point Energy (ZPE):   {zpe_ha:15.8f} Ha')
print(f'                             {zpe_ha * 27.2114:15.6f} eV')
print(f'                             {zpe_ha * 627.509:15.4f} kcal/mol')
print()

# Thermal corrections
h_corr = thermo_result['H_corr'][0]
print(f'  Thermal correction to H:   {h_corr:15.8f} Ha')
print(f'  (includes ZPE + vib + rot + trans + PV)')
print()

# Total enthalpy
h_total = thermo_result['H'][0]
print(f'  Total enthalpy H(298K):    {h_total:15.8f} Ha')
print()

# Entropy
S = thermo_result['S'][0]
print(f'  Entropy S(298K):           {S*1000:15.4f} J/mol/K')
print(f'                             {S*1000/4.184:15.4f} cal/mol/K')
print()

# Gibbs free energy
G = thermo_result['G'][0]
print(f'  Gibbs free energy G(298K): {G:15.8f} Ha')
print()

# ZPE-corrected energy
e_zpe = mf.e_tot + zpe_ha
print(f'  ZPE-corrected energy:      {e_zpe:15.8f} Ha')
print()

# Entropy breakdown
print('  Entropy contributions:')
if 'S_trans' in thermo_result:
    print(f"    Translational: {thermo_result['S_trans'][0]*1000:8.2f} J/mol/K")
if 'S_rot' in thermo_result:
    print(f"    Rotational:    {thermo_result['S_rot'][0]*1000:8.2f} J/mol/K")
if 'S_vib' in thermo_result:
    print(f"    Vibrational:   {thermo_result['S_vib'][0]*1000:8.2f} J/mol/K")

## 2. Vibrational Scaling Factors

Harmonic DFT frequencies systematically **overestimate** experimental fundamentals by ~3-5%
due to: (1) the harmonic approximation ignoring anharmonicity, and (2) the DFT functional's
imperfect potential energy surface.

**Recommended scaling factors** (multiply computed frequency by scale factor):

| Method / Basis | Scale Factor | Source |
|----------------|:------------:|--------|
| HF/6-31G\* | 0.8953 | NIST/CCCBDB |
| B3LYP/6-31G\* | 0.9614 | NIST/CCCBDB |
| B3LYP/def2-SVP | 0.9700 | Estimated |
| B3LYP/def2-TZVP | 0.9682 | Literature |
| PBE0/def2-SVP | 0.9660 | Literature |
| M06-2X/6-31G\* | 0.9523 | NIST/CCCBDB |
| ωB97X-D/6-31G\* | 0.9540 | Literature |
| TPSS/def2-TZVP | 0.9700 | Estimated |

**Example**: Unscaled B3LYP/def2-SVP O-H stretch ≈ 3905 cm⁻¹
After scaling (×0.97): 3788 cm⁻¹, closer to experimental 3756 cm⁻¹

**Reference database**: NIST Computational Chemistry Comparison and Benchmark Database  
(https://cccbdb.nist.gov) — extensive scaling factors for 100+ methods

## 3. Imaginary Frequencies and Transition States

### What is an imaginary frequency?

An **imaginary frequency** (displayed as negative in cm⁻¹) means the Hessian has a
negative eigenvalue at that geometry — the PES has negative curvature in that direction.

$$\omega_k^2 < 0 \implies \omega_k = i|\omega_k| \quad \text{(imaginary)}$$

This indicates the geometry is a **saddle point**, not a minimum.

### Interpreting imaginary frequencies:

| # Imaginary Modes | Geometry Type | Action |
|:-----------------:|---------------|--------|
| 0 | ✅ True minimum | Proceed with thermochemistry |
| 1 | 🔄 Transition state (TS) | Use IRC to find reactant/product |
| ≥2 | ❌ Higher-order saddle point | Re-optimize with perturbation |
| Small (<30i cm⁻¹) | ⚠️ Numerical noise | May be OK; check geometry |

### Transition State Theory

The rate constant for a reaction with barrier $\Delta G^\ddagger$ is given by:

$$k = \frac{k_B T}{h} e^{-\Delta G^\ddagger / RT}$$

This is the **Eyring equation** (Transition State Theory). A 1 kcal/mol error in
$\Delta G^\ddagger$ changes $k$ by a factor of ~5 at room temperature.

### IRC: Intrinsic Reaction Coordinate

To verify a TS connects the correct reactant and product, follow the **IRC**:
- Start at the TS geometry
- Perturb along the imaginary mode (forward and backward)
- Follow the steepest descent path to find reactant and product minima

In ORCA: `! IRC` keyword or `! NEB-TS` for nudged elastic band TS search.

## 🔬 Research Connection

Vibrational analysis is used throughout chemical research:

- **IR/Raman spectroscopy assignment**: Computed frequencies (×scaling factor) are compared
  with experimental spectra to assign peaks to specific molecular vibrations.
- **Thermochemistry**: Heats of formation, reaction free energies, and equilibrium constants
  all require ZPE and thermal corrections from frequency calculations.
- **Transition state validation**: Every published reaction mechanism includes a frequency
  calculation confirming the TS has exactly one imaginary frequency.
- **Astrochemistry**: Rotational and vibrational constants computed by DFT guide the
  assignment of new molecules detected in space by radio telescopes.

**Example**: The assignment of CO₂'s antisymmetric stretch at 2349 cm⁻¹ and its use
in climate science relies on DFT calculations confirming it is IR-active (unlike the
symmetric stretch at 1388 cm⁻¹ which is Raman-active).

## 📋 Summary

| Concept | Key Result |
|---------|----------|
| Hessian $H_{ij}$ | $\partial^2 E / \partial q_i \partial q_j$ at stationary point |
| Normal modes | Eigenvectors of mass-weighted Hessian |
| $3N-6$ vibrations | Non-linear molecule; $3N-5$ for linear |
| ZPE | $\frac{1}{2}\sum_k \hbar\omega_k$; always positive |
| IR activity | Requires $\partial\mu/\partial Q_k \neq 0$ |
| Scaling factor | ~0.97 for B3LYP/def2-SVP (harmonic overestimation) |
| Imaginary freq | Saddle point; 1 imaginary = transition state |
| Thermochemistry | Gibbs energy from $G = H - TS$ at 298.15 K |

**Always run a frequency calculation after geometry optimization!**

## 📝 Exercises

1. **CO₂ frequencies**: Compute vibrational frequencies for CO₂ at B3LYP/def2-SVP.
   - How many vibrational modes does CO₂ have? (linear molecule)
   - Which modes are IR active? Which are Raman active?
   - Compare computed frequencies with experimental: 667, 1388, 2349 cm⁻¹

2. **ZPE correction for H₂**: Compute the HF/cc-pVDZ energy and ZPE for H₂.
   - What is the ZPE? 
   - What fraction of the binding energy does the ZPE correction represent?

3. **Scaling factor test**: Apply the B3LYP/def2-SVP scaling factor (0.97) to the
   computed water frequencies. How much does agreement with experiment improve?

4. **Temperature dependence**: The `thermo.thermo()` function takes temperature as an
   argument. Compute G(T) for water at T = 200, 298, 400, 500, 600 K.
   Plot G vs T. At what temperature does G become more negative by 1 kcal/mol?

5. **Imaginary frequency exercise**: Take the optimized water geometry and manually 
   displace one H atom significantly (e.g., to r(OH) = 1.5 Å). Run a frequency 
   calculation. Do you get imaginary frequencies? Why?